#Importação das Bibliotecas

Estudos NLP: baixar embeddings, carregar vetores e usar Gensim.

In [29]:
!pip install gensim

Gensim é uma biblioteca do Python que ajuda o computador a entender textos.
Ela transforma palavras e documentos em números para descobrir padrões, sem precisar de muita configuração.

Em termos simples, ela serve para:

• Encontrar palavras parecidas.
• Agrupar textos por assunto.
• Transformar textos em vetores para usar em modelos de machine learning.

Ferramentas famosas dentro dela:

• Word2Vec aprende relações entre palavras.
• Doc2Vec representa textos completos como vetores.
• LDA descobre temas escondidos em um conjunto de textos.

A ideia central é: você fornece textos e ela devolve números que representam o significado. Isso facilita análises, buscas e modelos.

In [3]:
from huggingface_hub import hf_hub_download
from safetensors.numpy import load_file
from gensim.models import KeyedVectors
import numpy as np

• from huggingface_hub import hf_hub_download
Importa a função usada para baixar arquivos diretamente do Hugging Face Hub. É útil quando você quer carregar modelos, pesos ou dados sem baixar manualmente.

• from safetensors.numpy import load_file
Importa a função que lê arquivos no formato safetensors, um formato seguro e eficiente para armazenar pesos de modelos. Ela carrega o conteúdo como arrays NumPy.

• from gensim.models import KeyedVectors
Importa a classe KeyedVectors do Gensim, usada para trabalhar com embeddings de palavras. Ela permite carregar vetores já treinados e consultar similaridades, palavras mais próximas, etc.

• import numpy as np
Importa o NumPy, padrão na ciência de dados, para trabalhar com arrays e operações matemáticas necessárias para manipular os vetores.

# Carregamento do Modelo Pré-Treinado

`Modelo = Skipgram`

`Lingua = pt-br`

`Dimensionalidade = 100`

In [4]:
path = hf_hub_download(repo_id="nilc-nlp/word2vec-skip-gram-100d",
                       filename="embeddings.safetensors")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


embeddings.safetensors:   0%|          | 0.00/372M [00:00<?, ?B/s]

• repo_id="nilc-nlp/word2vec-skip-gram-100d" indica qual repositório você quer acessar. Nesse caso, é um modelo Word2Vec em português treinado com vetores de 100 dimensões.
• filename="embeddings.safetensors" diz qual arquivo dentro desse repositório você quer baixar.
• path = ... guarda o caminho do arquivo baixado no seu computador.

Depois dessa linha, a variável path contém o endereço do arquivo .safetensors que você poderá abrir e transformar em vetores para usar com Gensim ou NumPy.

Se quiser, posso mostrar o próximo passo para carregar esses vetores.

In [5]:
data = load_file(path)
vectors = data["embeddings"]

• data = load_file(path)
Abre o arquivo safetensors e devolve um dicionário com tudo que está dentro dele. É como abrir uma caixa com vários itens.

• vectors = data["embeddings"]
Pega dentro desse dicionário o item chamado embeddings, que é onde estão os vetores de palavras. O resultado costuma ser um array NumPy com as coordenadas numéricas dos vetores.

In [8]:
vocab_path = hf_hub_download(repo_id="nilc-nlp/word2vec-skip-gram-100d",
                             filename="vocab.txt")
with open(vocab_path) as f:
    vocab = [w.strip() for w in f]

print(vectors.shape)

vocab.txt: 0.00B [00:00, ?B/s]

(929606, 100)


• vocab_path = hf_hub_download(...) baixa o arquivo vocab.txt do mesmo repositório no Hugging Face. Esse arquivo contém todas as palavras que têm vetor associado.

• with open(vocab_path) as f: abre o arquivo.

• vocab = [w.strip() for w in f] cria uma lista com cada palavra do vocabulário, removendo quebras de linha. Agora você tem uma lista onde cada posição corresponde à mesma posição do vetor em vectors.

• print(vectors.shape) mostra o formato da matriz de embeddings. O padrão é algo como
(n_tokens, n_dimensoes)
ou seja, quantas palavras existem e qual o tamanho do vetor de cada uma.

929606 tokens, embeddings de dimensionalidade 100

In [11]:
vocab[0:10]

['</s>', ',', 'de', '.', 'a', 'o', 'e', 'que', 'do', 'da']

In [6]:
vectors

array([[-5.00000e-03, -4.75000e-04, -7.90000e-05, ..., -4.88400e-03,
        -4.31000e-03, -7.77000e-04],
       [-1.28430e-02, -1.19340e-01, -4.66080e-02, ..., -1.79057e-01,
        -1.03420e-02,  9.45570e-02],
       [-1.23121e-01,  4.72810e-02, -3.72836e-01, ...,  7.86110e-02,
        -4.23570e-02, -4.07120e-02],
       ...,
       [ 7.04850e-02, -1.51491e-01,  9.16230e-02, ..., -6.27100e-03,
        -5.33000e-03,  2.66000e-04],
       [-1.41350e-02, -1.05600e-03,  1.17900e-02, ..., -1.98200e-02,
        -1.17800e-03,  8.04000e-03],
       [-4.03820e-02,  2.21800e-02, -1.76100e-02, ..., -1.32100e-02,
         1.94280e-02,  3.28610e-02]], dtype=float32)

Verificando consistência

In [13]:
assert len(vocab) == vectors.shape[0]

Essa linha faz uma checagem de segurança no código.

assert len(vocab) == vectors.shape[0]
significa verificar se o número de tokens no vocabulário é igual ao número de vetores carregados.

• len(vocab) conta quantos tokens existem no vocab.txt.
• vectors.shape[0] é a quantidade de linhas da matriz de embeddings, cada linha sendo o vetor de um token.

Se os dois valores forem iguais, tudo continua normalmente.
Se forem diferentes, o Python interrompe o programa e mostra um erro, indicando que algo está inconsistente entre o vocabulário e os vetores.

É uma forma rápida de garantir que cada token tem exatamente um vetor correspondente.

# Criar KeyedVectors do Gensim

In [14]:
kv = KeyedVectors(vector_size=vectors.shape[1])
kv.add_vectors(vocab, vectors)

print("Modelo carregado!")
print("Dimensão:", kv.vector_size)
print("Vocabulário:", len(kv))

Modelo carregado!
Dimensão: 100
Vocabulário: 929606


Essas linhas criam um modelo do Gensim usando os vetores que você carregou e depois mostram algumas informações básicas. Explicando passo a passo de forma simples:

• kv = KeyedVectors(vector_size=vectors.shape[1])
Cria um objeto KeyedVectors vazio, dizendo qual é a dimensão dos vetores.
vectors.shape[1] é o número de colunas da matriz, ou seja, o tamanho de cada embedding.

• kv.add_vectors(vocab, vectors)
Adiciona ao modelo todos os tokens e seus respectivos vetores.
Cada item da lista vocab corresponde a uma linha na matriz vectors.

• print("Modelo carregado!")
Só confirma que deu tudo certo.

• print("Dimensão:", kv.vector_size)
Mostra quantas dimensões tem cada vetor.

• print("Vocabulário:", len(kv))
Mostra quantos tokens o modelo conhece.

Depois disso, você já consegue usar coisas como:

kv.most_similar("cachorro")
ou
kv.similarity("rei", "rainha").

# Análise dos Embeddings

## Busca de palavras mais similares

In [15]:
kv.most_similar("brasil")

[('japão', 0.7283652424812317),
 ('méxico', 0.7080685496330261),
 ('chile', 0.7040075063705444),
 ('mercosul', 0.6526188850402832),
 ('país', 0.6409318447113037),
 ('mundo', 0.6357104778289795),
 ('canadá', 0.6307356357574463),
 ('caribe', 0.6191968321800232),
 ('paraguai', 0.613277792930603),
 ('exterior', 0.6112302541732788)]

In [16]:
kv.most_similar("homem")

[('bandido', 0.7352811694145203),
 ('indivíduo', 0.7299154996871948),
 ('violador', 0.7252126336097717),
 ('garoto', 0.7196413278579712),
 ('andarilho', 0.7149757146835327),
 ('monstro', 0.7083200812339783),
 ('rapaz', 0.7063983082771301),
 ('mendigo', 0.6965662837028503),
 ('ancião', 0.6940868496894836),
 ('salteador', 0.693349301815033)]

In [17]:
kv.most_similar("mulher")

[('prostituta', 0.8479106426239014),
 ('amiga', 0.8421926498413086),
 ('menina', 0.836663007736206),
 ('mãe', 0.8169393539428711),
 ('garota', 0.8112452626228333),
 ('criança', 0.8063342571258545),
 ('rapariga', 0.8038724064826965),
 ('enteada', 0.7867975234985352),
 ('sogra', 0.7776657938957214),
 ('moça', 0.7772497534751892)]

In [18]:
kv.most_similar("medicina")

[('odontologia', 0.9035026431083679),
 ('psicologia', 0.8888339996337891),
 ('agronomia', 0.8718365430831909),
 ('faculdade', 0.8705843687057495),
 ('farmacologia', 0.8563330173492432),
 ('psiquiatria', 0.8418824076652527),
 ('microbiologia', 0.8349393606185913),
 ('neurologia', 0.8310769200325012),
 ('sociologia', 0.8252934813499451),
 ('biofísica', 0.8173642754554749)]

In [19]:
kv.most_similar("médico")

[('oftalmologista', 0.8561391830444336),
 ('anestesista', 0.8392991423606873),
 ('radiologista', 0.8357008099555969),
 ('cirurgião', 0.8351374864578247),
 ('ginecologista', 0.8290455937385559),
 ('ortopedista', 0.8279300928115845),
 ('urologista', 0.8211055397987366),
 ('neurologista', 0.8207941055297852),
 ('enfermeiro', 0.8162508606910706),
 ('cardiologista', 0.8099891543388367)]

In [20]:
kv.most_similar("médica")

[('odontológica', 0.7607213854789734),
 ('pediatria', 0.7437660694122314),
 ('clínica', 0.7410372495651245),
 ('psiquiátrica', 0.7279292941093445),
 ('ginecológica', 0.7236698269844055),
 ('clinica', 0.7213889360427856),
 ('multiprofissional', 0.7186266779899597),
 ('dermatologia', 0.7090435028076172),
 ('neurocirurgia', 0.7084410190582275),
 ('psiquiatria', 0.7072103023529053)]

In [21]:
kv.most_similar("dados")

[('cálculos', 0.8071094155311584),
 ('números', 0.7584335803985596),
 ('levantamentos', 0.7528680562973022),
 ('metadados', 0.7462222576141357),
 ('registos', 0.741081953048706),
 ('documentos', 0.7338613867759705),
 ('estatísticos', 0.7326570749282837),
 ('indicadores', 0.7155882120132446),
 ('atualizados', 0.7031537294387817),
 ('srtm', 0.6953601241111755)]

In [22]:
kv.most_similar("gato")

[('cão', 0.8402906060218811),
 ('papagaio', 0.7995700836181641),
 ('palhaço', 0.7928089499473572),
 ('cachorro', 0.7721726894378662),
 ('besouro', 0.7709585428237915),
 ('macaco', 0.7695572972297668),
 ('pássaro', 0.766366183757782),
 ('bichinho', 0.7639888525009155),
 ('cãozinho', 0.7616899609565735),
 ('porco-espinho', 0.7539454102516174)]

## Similaridade entre palavras

In [23]:
print(kv.similarity("homem", "mulher"))
print(kv.similarity("homem", "menino"))
print(kv.similarity("menina", "mulher"))
print(kv.similarity("homem", "computador"))

0.4481777
0.6853265
0.83666307
0.18459874


In [24]:
print(kv.similarity("mulher", "inteligente"))
print(kv.similarity("homem", "inteligente"))
print(kv.similarity("mulher", "competencia"))
print(kv.similarity("homem", "competencia"))

0.27615368
0.34215763
0.13466354
-0.0028040358


## Analogias

Rei está para homem assim como X está pra mulher

In [25]:
kv.most_similar(positive=["rei", "mulher"], negative=["homem"], topn=10)

[('rainha-consorte', 0.7912214994430542),
 ('primogénita', 0.7738461494445801),
 ('imperatriz-mãe', 0.7646884322166443),
 ('paleóloga', 0.752788245677948),
 ('dama-de-companhia', 0.7478024363517761),
 ('consorte', 0.7475904226303101),
 ('princesa-eleitora', 0.7472771406173706),
 ('piroska', 0.7468665838241577),
 ('ulrica', 0.7454056143760681),
 ('ranavalona', 0.7441918253898621)]

In [26]:
kv.most_similar(positive=["médico", "mulher"], negative=["homem"], topn=10)

[('enfermeira', 0.7504610419273376),
 ('clínica', 0.7258327007293701),
 ('nutricionista', 0.7182406187057495),
 ('médica', 0.7078936696052551),
 ('clinica', 0.700611412525177),
 ('obstetra', 0.6919179558753967),
 ('fonoaudióloga', 0.6907674074172974),
 ('psicóloga', 0.6823008060455322),
 ('fisioterapeuta', 0.6814467906951904),
 ('cirurgiã', 0.6747321486473083)]

In [27]:
kv.most_similar(positive=["médica", "homem"], negative=["mulher"], topn=10)

[('médico', 0.6110728979110718),
 ('perito', 0.5980631709098816),
 ('estudo', 0.5885353684425354),
 ('investigador', 0.5560773015022278),
 ('estomatologista', 0.5480618476867676),
 ('clinico', 0.5468452572822571),
 ('laudo', 0.5377167463302612),
 ('avaliador', 0.5317424535751343),
 ('exame', 0.5314248204231262),
 ('forense', 0.5311265587806702)]

In [28]:
kv.most_similar(positive=["vendedor", "futebol"], negative=["vendas"], topn=10)

[('ex-jogador', 0.6733699440956116),
 ('clube', 0.6509010791778564),
 ('amador', 0.6463962197303772),
 ('ex-árbitro', 0.6364107728004456),
 ('emiratense', 0.6252110600471497),
 ('céres', 0.6209861636161804),
 ('ex-esportista', 0.617440402507782),
 ('jogador', 0.6046038866043091),
 ('remador', 0.6022697687149048),
 ('rúgbi', 0.6002370715141296)]

In [ ]:
kv.most_similar(positive=["estudei", "trabalho"], negative=["ensino"], topn=10)

[('aprecio', 0.5707091093063354),
 ('trabalhei', 0.5696157813072205),
 ('acompanhei', 0.5587385892868042),
 ('gosto', 0.5579565763473511),
 ('experimentei', 0.5568064451217651),
 ('vivi', 0.5542948842048645),
 ('interessou-me', 0.5522068738937378),
 ('melhorei', 0.5521753430366516),
 ('revi', 0.5415771007537842),
 ('hesitei', 0.5366280674934387)]